# Compressing UFLD-v1 (ResNet-18 / TuSimple) for Raspberry Pi 5

This notebook uses the `ufld_compress` framework to compress **Ultra-Fast-Lane-Detection v1** (the `cfzd` repository) with three families of methods, and evaluates every variant in terms of
**accuracy / F1 / FP / FN, number of parameters, file size, MACs/FLOPs/BOPs, latency (1, 4 and 6 threads), memory, energy** and agreement with the original model.
All runs are logged to **MLflow**; training curves and **PyTorch Profiler** traces are written to **TensorBoard**.

| Section | Methods |
|---|---|
| 1. Quantization | PyTorch dynamic (INT8 / FP16), static PTQ with FX (different observers, per-channel, mixed precision, qnnpack engine), QAT + KD, ONNX Runtime dynamic and static (MinMax / Entropy / Percentile) |
| 2. Pruning | Unstructured (global / layer-wise / random), structured channel pruning (L1 / BN-γ / Taylor / random; block-internal channels, residual channels, FC hidden layer), BN recalibration, gradual AGP |
| 3. Knowledge distillation | Logit KD, location KD (UFLD-specific), feature KD (FitNets), attention transfer, label-free KD, MobileNet student |
| 4. Combined | Prune → KD → INT8 (PTQ / QAT / ONNX Runtime) → final model for Pi 5 |

**Key fact about this model:** of its ~61 M parameters, ~50 M are in the FC head (`Linear(2048 → 101·56·4)`), but more than 99 % of the compute is in the backbone.
So to reduce **size / memory** the FC head must be quantized or pruned, and to reduce **latency / energy** the backbone must be (channels and INT8).

### Run time on a Ryzen 5 5500U
With `PROFILE = "quick"` the whole notebook takes roughly 1.5–2.5 hours (mostly fine-tuning). `"full"` gives more accurate results but is several times slower.
Each section is independent: after running the baseline section, any other section can be run on its own.

## 0. Installation and setup

```bash
python -m venv .venv && source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
pip install -r requirements.txt
```

**Data:** the TuSimple dataset (train_set and test_set, plus `test_label.json`). Folder layout as in the UFLD instructions:
```
data/tusimple/
  clips/...                      (train and test images side by side)
  label_data_0313.json  label_data_0531.json  label_data_0601.json
  test_label.json
```
Running `scripts/convert_tusimple.py` is **not** needed; the labels are generated in memory with the same logic.

**Weights:** `tusimple_18.pth` from the README of cfzd/Ultra-Fast-Lane-Detection → `weights/tusimple_18.pth`.

If the data or the weights are not found, the notebook automatically runs in **DRY_RUN** mode (synthetic data + random weights) so the whole pipeline can be tested; accuracy numbers are meaningless in that mode.

In [ ]:
# %pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu
# %pip install -q -r ../requirements.txt

## 1. Configuration

In [ ]:
import sys, os, copy, time, json, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 250)

from ufld_compress.config import Config
from ufld_compress import model as M, data as D, quantization as Q, pruning as PR, distillation as KD, profiling as P
from ufld_compress.benchmark import Benchmarker
from ufld_compress.tracking import Tracker
from ufld_compress.utils import set_threads, seed_everything, system_info, default_quant_backend

PROFILE = "quick"            # "quick" | "full"
DATA_ROOT  = ROOT / "data" / "tusimple"
CHECKPOINT = ROOT / "weights" / "tusimple_18.pth"
OUT = ROOT / "outputs"

DRY_RUN = not (CHECKPOINT.exists() and (DATA_ROOT / "test_label.json").exists())
cfg = Config(data_root=str(DATA_ROOT), checkpoint=str(CHECKPOINT), output_dir=str(OUT),
             num_threads=6,      # 5500U: 6 physical cores (SMT threads do not help GEMMs much)
             num_workers=2,      # DataLoader workers
             dry_run=DRY_RUN)

S = {
    # eval_images: subset of the 2782 test images (None = all) | train_images: subset of 3626 train images
    "quick": dict(eval_images=300,  train_images=None, batch=4, ft_steps=150, kd_steps=150, qat_steps=100,
                  agp_steps=200, calib_batches=16, ort_calib_images=8, lat_iters=20, energy_iters=15),
    "full":  dict(eval_images=None, train_images=None, batch=8, ft_steps=1500, kd_steps=1500, qat_steps=600,
                  agp_steps=2000, calib_batches=32, ort_calib_images=32, lat_iters=50, energy_iters=40),
}[PROFILE]
if DRY_RUN:
    S.update(eval_images=16, train_images=32, ft_steps=4, kd_steps=4, qat_steps=4, agp_steps=6,
             calib_batches=2, ort_calib_images=2, lat_iters=3, energy_iters=3)
    print("⚠️  DRY_RUN: dataset/checkpoint not found -> synthetic data + random weights")

set_threads(cfg.num_threads); seed_everything(cfg.seed)
QBACKEND = default_quant_backend()       # 'x86' on the Ryzen, 'qnnpack' on the Pi
print("quantization engine:", QBACKEND, "| available:", torch.backends.quantized.supported_engines)
print(json.dumps(system_info(), indent=1))

### MLflow and TensorBoard
- MLflow UI (in a terminal, in the project folder): `mlflow ui --backend-store-uri sqlite:///mlflow.db` → http://127.0.0.1:5000
  In the UI you can filter runs (`tags.group = "pruning"`), select several runs and **Compare** them (parallel-coordinates / scatter plots between parameters and metrics).
- TensorBoard: `tensorboard --logdir outputs/tensorboard` (training curves under `train/`, the HParams tab, and profiler traces under `profile/`, which can also be opened with Perfetto).

In [ ]:
tracker = Tracker("ufld-tusimple-compression" + ("-dryrun" if DRY_RUN else ""),
                  tracking_uri="sqlite:///" + (ROOT / "mlflow.db").as_posix(),
                  artifact_dir=str(ROOT / "mlartifacts"))
# %load_ext tensorboard
# %tensorboard --logdir ../outputs/tensorboard

## 2. Data

In [ ]:
train_ds, test_ds = D.build_datasets(cfg, train_subset=S["train_images"], test_subset=S["eval_images"], augment=True)
# calibration / BN-recalibration images come from the TRAIN set without augmentation (never from the test set)
calib_ds = D.SyntheticTrainDataset(cfg, 32) if DRY_RUN else D.TuSimpleTrainDataset(cfg, subset=256, augment=False, seed=1)

test_loader  = D.make_loader(test_ds,  batch_size=4,          num_workers=cfg.num_workers)
train_loader = D.make_loader(train_ds, batch_size=S["batch"], shuffle=True, num_workers=cfg.num_workers, drop_last=True)
calib_loader = D.make_loader(calib_ds, batch_size=4,          num_workers=cfg.num_workers)
calib1_loader = D.make_loader(calib_ds, batch_size=1,         num_workers=0)   # ONNX Runtime calibration

def calib(n_batches, loader=calib_loader):
    "fresh generator of n image batches"
    return D.images_only(loader, n_batches)

print(f"train={len(train_ds)}  test(eval)={len(test_ds)}  calib={len(calib_ds)}")
x, y = next(iter(train_loader)); print("batch:", tuple(x.shape), "labels:", tuple(y.shape), "classes 0..", cfg.griding_num)

## 3. Baseline model (FP32) and the Benchmarker
`Benchmarker.run` does the following for every variant and logs everything to one MLflow run:
1. Official TuSimple evaluation (accuracy, FP, FN, F1) + **teacher_agreement** (fraction of row/lane cells whose prediction is identical to the original model's)
2. Parameters (total and non-zero), state_dict size and its zlib-compressed size, size of the deployable artifact (TorchScript / ONNX)
3. MACs, FLOPs, effective MACs (accounting for sparsity), BOPs (MACs × bitsᵂ × bitsᴬ)
4. Latency with 1, 4 (= number of Pi 5 cores) and 6 threads: mean, p90, FPS
5. Energy per inference (RAPL on Linux, PMIC on Pi 5, otherwise an estimate from utilization and TDP; the method is recorded in `energy_method`)
6. Memory: RSS increase during inference, and an **isolated** measurement in a fresh process (weights, peak during execution)
7. PyTorch Profiler → TensorBoard + table of the most expensive operators (artifact in MLflow)

In [ ]:
teacher = M.build_teacher(cfg)
bench = Benchmarker(cfg, test_loader, test_ds, tracker, out_dir=str(OUT),
                    latency_threads=(1, 4, 6), latency_iters=S["lat_iters"], energy_iters=S["energy_iters"])

# where do the parameters and the compute live?
rows = []
for name, mod in [("backbone", teacher.model), ("pool", teacher.pool), ("cls.0 (1800→2048)", teacher.cls[0]),
                  ("cls.2 (2048→22624)", teacher.cls[2])]:
    macs = P.count_macs(mod, (1, 512, 9, 25) if name == "pool" else (1, 1800) if name.startswith("cls.0")
                        else (1, 2048) if name.startswith("cls.2") else cfg.input_shape)["macs_g"]
    rows.append((name, sum(p.numel() for p in mod.parameters()), macs))
pt = pd.DataFrame(rows, columns=["module", "params", "GMACs"])
pt["params_%"] = 100 * pt.params / pt.params.sum(); pt["MACs_%"] = 100 * pt.GMACs / pt.GMACs.sum()
display(pt)
print("torch FlopCounter GFLOPs:", P.torch_flop_counter(teacher, cfg.input_shape))

In [ ]:
base = bench.run("fp32_baseline", teacher, group="baseline", params={"method": "none", "precision": "fp32"},
                 is_reference=True)
pf = OUT / "profiles" / "fp32_baseline.txt"
print(pf.read_text()[:3000] if pf.exists() else "(profiler output missing)")

### Model output (green: prediction, red: ground truth)

In [ ]:
import cv2
from ufld_compress.lane_utils import logits_to_lanes
from ufld_compress.runners import TorchRunner, OrtRunner

def show_predictions(runner, n=3, title=""):
    if DRY_RUN:
        print("(dry run: no images)"); return
    fig, axs = plt.subplots(1, n, figsize=(6 * n, 3.6))
    for ax, k in zip(np.atleast_1d(axs), range(n)):
        x, i = test_ds[k * 7 % len(test_ds)]
        out = runner(x[None])[0]
        img = cv2.cvtColor(cv2.imread(test_ds.image_path(i)), cv2.COLOR_BGR2RGB)
        gt = test_ds.items[i]
        for lane in gt["lanes"]:
            for xx, yy in zip(lane, gt["h_samples"]):
                if xx >= 0: cv2.circle(img, (xx, yy), 4, (255, 0, 0), -1)
        for lane in logits_to_lanes(out, cfg.griding_num, cfg.img_w):
            for xx, yy in zip(lane, gt["h_samples"]):
                if xx >= 0: cv2.circle(img, (xx, yy), 3, (0, 255, 0), -1)
        ax.imshow(img); ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

show_predictions(TorchRunner(teacher), title="FP32 baseline")

## 4. Quantization

### 4.1 Dynamic quantization (PyTorch)
Only the `Linear` layers (FC head) get INT8 (or FP16) weights; activations are quantized on the fly. No calibration data needed. Large size reduction because 82 % of the parameters are in the FC head, but latency barely changes (the compute is in the convolutions).

In [ ]:
for dt_name, dt in [("int8", torch.qint8), ("fp16", torch.float16)]:
    qm = Q.dynamic_quantize(teacher, dtype=dt, backend=QBACKEND)
    bench.run(f"q_dynamic_{dt_name}", qm, group="quantization",
              params={"method": "dynamic", "dtype": dt_name, "layers": "Linear", "engine": QBACKEND})

### 4.2 Static post-training quantization with FX (full INT8)
Conv+BN+ReLU are fused automatically and activations are quantized using calibration data (from the train set). Variables tested:
- activation **observer**: `minmax` / `histogram` (minimizes quantization error) / `moving_avg`
- **per-channel** vs per-tensor weights
- **mixed precision**: keep the first and last layers in FP32 (the most sensitive layers)
- the **qnnpack** engine (the one used on the Pi 5) to check accuracy; its latency on x86 is not representative.

With `reduce_range=True` on x86, activations use 7 bits to avoid overflow on CPUs without VNNI (such as the 5500U).

In [ ]:
ptq_variants = [
    dict(act_observer="minmax",    per_channel=False),
    dict(act_observer="minmax",    per_channel=True),
    dict(act_observer="histogram", per_channel=True),
    dict(act_observer="histogram", per_channel=True, skip_modules=("model.conv1", "cls.2")),   # mixed precision
]
for v in ptq_variants:
    name = f"q_ptq_{v['act_observer']}_{'pc' if v['per_channel'] else 'pt'}" + ("_mixed" if v.get("skip_modules") else "")
    qm = Q.static_quantize_fx(teacher, calib(S["calib_batches"]), S["calib_batches"], backend=QBACKEND,
                              input_shape=cfg.input_shape, **v)
    bench.run(name, qm, group="quantization",
              params={"method": "ptq_fx", "engine": QBACKEND, "calib_batches": S["calib_batches"], **v})

if "qnnpack" in torch.backends.quantized.supported_engines and QBACKEND != "qnnpack":
    qm = Q.static_quantize_fx(teacher, calib(S["calib_batches"]), S["calib_batches"], backend="qnnpack",
                              act_observer="histogram", per_channel=False, input_shape=cfg.input_shape)
    bench.run("q_ptq_qnnpack_pi_engine", qm, group="quantization",
              params={"method": "ptq_fx", "engine": "qnnpack", "act_observer": "histogram", "per_channel": False,
                      "note": "accuracy check for the ARM engine; x86 latency not representative"})

### 4.3 Quantization-aware training + distillation
FakeQuant modules are inserted into the graph and the model is fine-tuned for a few hundred steps with **KD from the FP32 model** (plus labels). BN is frozen and the observers are frozen at 70 % of training.

In [ ]:
def finetune(student, run_name, tc, teacher_model=None, callbacks=()):
    tr = KD.Trainer(student, train_loader, tc, teacher=teacher_model,
                    tb_logdir=str(OUT / "tensorboard" / "train" / run_name), callbacks=callbacks)
    t0 = time.time(); hist = tr.fit()
    print(f"{run_name}: {len(hist)} steps in {time.time() - t0:.0f}s, final loss {hist[-1]['loss']:.4f}")
    return student.eval(), hist

qat = Q.prepare_qat(teacher, backend=QBACKEND, input_shape=cfg.input_shape)
tc_qat = KD.TrainConfig(max_steps=S["qat_steps"], lr=1e-5, warmup_steps=10, hard_weight=1.0, kd_weight=1.0,
                        temperature=2.0, freeze_bn=True)
qat, hist = finetune(qat, "q_qat_kd", tc_qat, teacher,
                     callbacks=[Q.qat_step_callback(freeze_observer_step=int(0.7 * S["qat_steps"]))])
qm = Q.convert_qat(qat)
bench.run("q_qat_kd", qm, group="quantization", train_history=hist,
          params={"method": "qat_fx", "engine": QBACKEND, **tc_qat.to_dict()})

### 4.4 ONNX Runtime (recommended deployment path for Raspberry Pi 5)
ONNX Runtime has optimized INT8 kernels on ARM64 and does not need PyTorch on the Pi.
- **Dynamic**: MatMul/Gemm only (FC head)
- **Static QDQ**: all Conv/Gemm, calibrated with `minmax`, `entropy` (KL) and `percentile`

⚠️ Entropy/Percentile calibration in ORT keeps all activations in RAM (~100 MB per image); keep the number of calibration images in line with your RAM.

In [ ]:
ONNX = OUT / "onnx"; ONNX.mkdir(parents=True, exist_ok=True)
onnx_fp32 = Q.export_onnx(teacher, str(ONNX / "teacher_fp32.onnx"), cfg.input_shape)
bench.run("ort_fp32", onnx_path=onnx_fp32, ref_model=teacher, group="quantization",
          params={"method": "onnxruntime", "precision": "fp32"})

p = Q.ort_quantize_dynamic(onnx_fp32, str(ONNX / "teacher_int8_dynamic.onnx"))
bench.run("ort_int8_dynamic", onnx_path=p, ref_model=teacher, group="quantization",
          params={"method": "ort_dynamic", "ops": "MatMul,Gemm"})

for method in ["minmax", "entropy", "percentile"]:
    p = Q.ort_quantize_static(onnx_fp32, str(ONNX / f"teacher_int8_static_{method}.onnx"),
                              calib(S["ort_calib_images"], calib1_loader), S["ort_calib_images"],
                              method=method, per_channel=True)
    bench.run(f"ort_int8_static_{method}", onnx_path=p, ref_model=teacher, group="quantization", bits=8,
              params={"method": "ort_static_qdq", "calibration": method, "per_channel": True,
                      "calib_images": S["ort_calib_images"]})

In [ ]:
qcols = ["name", "accuracy", "f1", "teacher_agreement", "artifact_mb", "lat_ms_t1", "lat_ms_t4", "energy_mj_per_inf", "mem_runtime_mb"]
t = bench.table(); display(t[[c for c in qcols if c in t]].round(4))

## 5. Pruning

### 5.1 Unstructured (no fine-tuning)
Small weights are set to zero. `global` uses one threshold for the whole network (in practice it prunes mostly the FC head), `layer` prunes every layer by the same amount, and `random` is the baseline.
On a dense CPU the speed does not change; the main gains are the **compressed size (`size_compressed_mb`)** and the effective MACs.

In [ ]:
unstructured = [("global", "l1", 0.5), ("global", "l1", 0.7), ("global", "l1", 0.9),
                ("layer", "l1", 0.5), ("global", "random", 0.5)]
if PROFILE == "full":
    unstructured += [("global", "l1", 0.3), ("layer", "l1", 0.7)]
for scope, method, s in unstructured:
    m = PR.unstructured_prune(teacher, s, scope=scope, method=method)
    bench.run(f"prune_unstruct_{scope}_{method}_{int(s * 100)}", m, group="pruning",
              params={"method": "unstructured", "scope": scope, "criterion": method, "sparsity": s, "finetune": 0})

### 5.2 Structured channel pruning: comparing importance criteria (one-shot + BN recalibration)
Channels are physically removed → FLOPs, latency and memory really go down. Groups:
- **block-internal** channels of every BasicBlock (`conv1→conv2`)
- **residual** channels of each stage (all blocks of a stage must be pruned together), via `layer_ratios={"model.stage": r}`
- **FC** hidden units (2048), the largest effect on the parameter count

Criteria: `l1`, `bn_gamma` (Network Slimming), `taylor` (first-order estimate of the loss change caused by removal), `random`.
After pruning, the BatchNorm statistics are re-estimated (cheap and effective).

In [ ]:
taylor_src = copy.deepcopy(teacher)
taylor_scores = PR.compute_taylor_scores(taylor_src, calib(8), lambda m, b: PR.pseudo_label_loss(m, b), n_batches=8)

# pruning levels used in 5.3, 6 and 7  (ratio: block-internal channels, residual: stage channels, fc: hidden FC units)
prune_cfgs = {
    "light":      dict(ratio=0.3, residual=0.0,  fc=0.5,   criterion="l1"),
    "medium":     dict(ratio=0.5, residual=0.25, fc=0.75,  criterion="taylor"),
    "aggressive": dict(ratio=0.6, residual=0.5,  fc=0.875, criterion="taylor", strategy="global"),
}

def make_pruned(ratio, residual=0.0, fc=0.75, criterion="l1", strategy="uniform", recal=True, src=None):
    src = src if src is not None else (taylor_src if criterion == "taylor" else teacher)
    m = PR.channel_prune(src, ratio=ratio, criterion=criterion, prune_residual=residual > 0, fc_ratio=fc,
                         strategy=strategy, layer_ratios={"model.stage": residual} if residual > 0 else None,
                         taylor_scores=taylor_scores if criterion == "taylor" else None,
                         input_shape=cfg.input_shape)
    if recal:
        PR.recalibrate_bn(m, calib(20), 20)
    return m

for crit in ["l1", "bn_gamma", "taylor", "random"]:
    m = make_pruned(0.5, residual=0.0, fc=0.75, criterion=crit)
    bench.run(f"prune_struct_{crit}_r50_fc75_oneshot", m, group="pruning",
              params={"method": "channel", "criterion": crit, "ratio": 0.5, "residual": 0, "fc_ratio": 0.75,
                      "bn_recal": True, "finetune": 0, "report": m.prune_report})

# effect of BN recalibration alone
m = make_pruned(0.5, fc=0.75, criterion="l1", recal=False)
bench.run("prune_struct_l1_r50_fc75_no_bnrecal", m, group="pruning",
          params={"method": "channel", "criterion": "l1", "ratio": 0.5, "fc_ratio": 0.75, "bn_recal": False, "finetune": 0})

### 5.3 Structured pruning + accuracy recovery with KD
Three compression levels, each fine-tuned for `ft_steps` steps with labels + logit KD + location KD from the original model.

In [ ]:
tc_ft = KD.TrainConfig(max_steps=S["ft_steps"], lr=2e-4, warmup_steps=10, hard_weight=1.0, kd_weight=1.0,
                       temperature=4.0, loc_weight=1.0)
pruned_models = {}
for level, pc in prune_cfgs.items():
    m = make_pruned(**pc)
    m, hist = finetune(m, f"prune_struct_{level}_kd", tc_ft, teacher)
    pruned_models[level] = m
    M.save_model(m, str(OUT / "models" / f"pruned_{level}_kd.pth"))
    bench.run(f"prune_struct_{level}_kd", m, group="pruning", train_history=hist,
              params={"method": "channel+kd", **pc, "report": m.prune_report, **tc_ft.to_dict()})

### 5.4 Gradual magnitude pruning (AGP) during KD
Sparsity increases cubically from 0 to 80 % during training (Zhu & Gupta, 2017), and the model recovers with KD at every step.

In [ ]:
agp_model = copy.deepcopy(teacher)
agp = PR.AGPPruner(agp_model, final_sparsity=0.8, start_step=0, end_step=int(0.75 * S["agp_steps"]),
                   frequency=max(1, S["agp_steps"] // 20))
tc_agp = KD.TrainConfig(max_steps=S["agp_steps"], lr=1e-4, warmup_steps=10, hard_weight=1.0, kd_weight=1.0, temperature=4.0)
agp_model, hist = finetune(agp_model, "prune_agp80_kd", tc_agp, teacher, callbacks=[agp])
PR.make_pruning_permanent(agp_model)
print("sparsity:", PR.sparsity_report(agp_model)["__global__"])
bench.run("prune_agp80_kd", agp_model, group="pruning", train_history=hist,
          params={"method": "agp+kd", "final_sparsity": 0.8, "schedule": agp.history, **tc_agp.to_dict()})

## 6. Knowledge distillation: comparing methods on a fixed student
Student: the "medium" pruned ResNet-18 (before fine-tuning). All variants use the same number of steps:

| Name | Loss |
|---|---|
| `hard_only` | labels only (focal + relation), no KD |
| `kd_only` | logit KD only, **no labels** (usable with unlabeled video) |
| `hard+logit` | labels + KL on the 101-cell distribution of every row/lane |
| `hard+logit+loc` | + L1 distance between the expected lane positions (UFLD-specific) |
| `hard+logit+feat` | + FitNets on layer2/3/4 with 1×1 adapters |
| `hard+logit+at` | + attention transfer |

In [ ]:
kd_student_init = make_pruned(**prune_cfgs["medium"])
kd_variants = {
    "hard_only":       dict(hard_weight=1.0, kd_weight=0.0),
    "kd_only":         dict(hard_weight=0.0, kd_weight=1.0),
    "hard+logit":      dict(hard_weight=1.0, kd_weight=1.0),
    "hard+logit+loc":  dict(hard_weight=1.0, kd_weight=1.0, loc_weight=1.0),
    "hard+logit+feat": dict(hard_weight=1.0, kd_weight=1.0, feat_weight=1.0),
    "hard+logit+at":   dict(hard_weight=1.0, kd_weight=1.0, at_weight=200.0),
}
for kd_name, w in kd_variants.items():
    tc = KD.TrainConfig(max_steps=S["kd_steps"], lr=2e-4, warmup_steps=10, temperature=4.0, **w)
    uses_teacher = any(w.get(k, 0) for k in ("kd_weight", "loc_weight", "feat_weight", "at_weight"))
    st, hist = finetune(copy.deepcopy(kd_student_init), f"kd_{kd_name}", tc, teacher if uses_teacher else None)
    bench.run(f"kd_{kd_name}", st, group="distillation", train_history=hist,
              params={"method": "kd", "student": "resnet18_pruned_medium", "kd_variant": kd_name, **tc.to_dict()})

### (Optional) MobileNet student from ImageNet weights
A completely different backbone (MobileNetV2/V3 + an FC head with 1024 units) that learns only from the teacher. Because it is trained from scratch on TuSimple, good results need
several thousand steps (several hours on a 5500U). Off by default.

In [ ]:
RUN_MOBILENET = False
if RUN_MOBILENET or DRY_RUN:
    arch = "mobilenet_v2"
    st = M.build_student(cfg, arch, pretrained=True, fc_hidden=1024)
    steps = S["kd_steps"] * (10 if PROFILE == "full" and not DRY_RUN else 1)
    tc = KD.TrainConfig(max_steps=steps, lr=5e-4, warmup_steps=50 if not DRY_RUN else 1, hard_weight=1.0,
                        kd_weight=1.0, loc_weight=1.0, at_weight=200.0, freeze_bn=False, temperature=4.0)
    st, hist = finetune(st, f"kd_{arch}", tc, teacher)
    bench.run(f"kd_student_{arch}", st, group="distillation", train_history=hist,
              params={"method": "kd", "student": arch, **tc.to_dict()})

## 7. Combined pipeline: Prune → KD → INT8
The best pruned model (default: "medium") is then quantized. Three paths:
1. PyTorch FX PTQ
2. QAT + KD on the pruned model
3. ONNX Runtime static INT8 (the final file for Pi 5)

In [ ]:
best_level = "medium"          # change after looking at section 5.3
pm = pruned_models[best_level]   # from section 5.3 (or: M.load_model(OUT / "models" / f"pruned_{best_level}_kd.pth"))
tc_qat = KD.TrainConfig(max_steps=S["qat_steps"], lr=1e-5, warmup_steps=10, hard_weight=1.0, kd_weight=1.0,
                        temperature=2.0, freeze_bn=True)

qm = Q.static_quantize_fx(pm, calib(S["calib_batches"]), S["calib_batches"], backend=QBACKEND,
                          act_observer="histogram", per_channel=True, input_shape=cfg.input_shape)
bench.run(f"combo_{best_level}_kd_ptq_int8", qm, group="combined",
          params={"pipeline": "prune+kd+ptq_fx", "prune": prune_cfgs[best_level], "engine": QBACKEND})

qat = Q.prepare_qat(pm, backend=QBACKEND, input_shape=cfg.input_shape)
qat, hist = finetune(qat, f"combo_{best_level}_kd_qat", tc_qat, teacher,
                     callbacks=[Q.qat_step_callback(freeze_observer_step=int(0.7 * S["qat_steps"]))])
bench.run(f"combo_{best_level}_kd_qat_int8", Q.convert_qat(qat), group="combined", train_history=hist,
          params={"pipeline": "prune+kd+qat", "prune": prune_cfgs[best_level], **tc_qat.to_dict()})

DEPLOY = OUT / "deploy"; DEPLOY.mkdir(exist_ok=True)
for level, m in pruned_models.items():
    f32 = Q.export_onnx(m, str(DEPLOY / f"ufld_pruned_{level}_fp32.onnx"), cfg.input_shape)
    for method in (["minmax", "percentile"] if level == best_level else ["percentile"]):
        p = Q.ort_quantize_static(f32, str(DEPLOY / f"ufld_pruned_{level}_int8_{method}.onnx"),
                                  calib(S["ort_calib_images"], calib1_loader), S["ort_calib_images"],
                                  method=method, per_channel=True)
        bench.run(f"combo_{level}_kd_ort_int8_{method}", onnx_path=p, ref_model=m, group="combined", bits=8,
                  params={"pipeline": "prune+kd+ort_static", "prune": prune_cfgs[level], "calibration": method})
Q.export_onnx(teacher, str(DEPLOY / "ufld_teacher_fp32.onnx"), cfg.input_shape)
print(sorted(p.name for p in DEPLOY.glob("*.onnx")))

## 8. Final comparison

In [ ]:
df = bench.table()
bench.save("results")
show = ["name", "group", "accuracy", "acc_drop", "f1", "teacher_agreement", "params_total", "sparsity", "artifact_mb",
        "compression_x", "macs_g", "lat_ms_t4", "speedup_t4", "mem_runtime_mb", "energy_mj_per_inf"]
display(df[[c for c in show if c in df]].sort_values(["group", "lat_ms_t4"]).round(4))

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(16, 11))
bench.pareto_plot("lat_ms_t4", "accuracy", ax=axs[0, 0], title="Accuracy vs latency (4 threads ≈ Pi 5 cores)")
bench.pareto_plot("artifact_mb", "accuracy", logx=True, ax=axs[0, 1], title="Accuracy vs model file size")
if "energy_mj_per_inf" in df:
    bench.pareto_plot("energy_mj_per_inf", "accuracy", ax=axs[1, 0], title="Accuracy vs energy / inference")
if "mem_runtime_mb" in df:
    bench.pareto_plot("mem_runtime_mb", "accuracy", ax=axs[1, 1], title="Accuracy vs runtime memory")
plt.tight_layout(); fig.savefig(OUT / "pareto.png", dpi=130); plt.show()

with tracker.run("summary", tags={"group": "summary"}):
    tracker.log_artifact(OUT / "results.csv"); tracker.log_artifact(OUT / "results_full.csv")
    tracker.log_figure(fig, "pareto.png")

In [ ]:
# pick the most accurate model under a latency budget (4 threads on the Ryzen ~ an upper bound for Pi speed-up ratios)
budget_ms = float(df["lat_ms_t4"].min() * 2.0)
cands = df[df["lat_ms_t4"] <= budget_ms].sort_values("accuracy", ascending=False)
display(cands[["name", "accuracy", "lat_ms_t4", "artifact_mb", "energy_mj_per_inf"]].head(5))

## 9. Deployment and measurement on Raspberry Pi 5
Latency and energy on x86 are only useful for relative comparison; take the final numbers on the Pi itself:

```bash
# On the Pi 5 (Raspberry Pi OS 64-bit)
python3 -m venv ~/ufld && source ~/ufld/bin/activate
pip install onnxruntime numpy opencv-python-headless psutil mlflow   # mlflow is optional
scp -r outputs/deploy scripts ufld_compress  pi@raspberrypi.local:~/ufld_bench/

# On the laptop, start an MLflow server so the Pi results land in the same experiment:
mlflow server --backend-store-uri sqlite:///mlflow.db --artifacts-destination ./mlartifacts --host 0.0.0.0 --port 5000

# On the Pi:
cd ~/ufld_bench
python scripts/pi5_benchmark.py --models deploy/*.onnx --threads 1 2 4 --iters 100 \
       --mlflow-uri http://<LAPTOP-IP>:5000 --experiment ufld-tusimple-compression
python scripts/pi5_benchmark.py --models deploy/ufld_pruned_medium_int8_percentile.onnx --demo video.mp4
```
On the Pi 5, energy is read from the PMIC (`vcgencmd pmic_read_adc`). For stable numbers: active cooling, governor set to `performance`
(`echo performance | sudo tee /sys/devices/system/cpu/cpu*/cpufreq/scaling_governor`) and the official 27 W power supply.